In [4]:
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams, PointStruct

import pandas as pd
import openai

### Read the sampled dataset

In [5]:
df_items = pd.read_json('../../data/meta_Electronics_filtered_2022_2023_with_category_ratings_100_sample_1000.jsonl', lines=True)

In [6]:
df_items.head()

,main_category,title,average_rating,rating_number,features,description,price,images,videos,store,categories,details,parent_asin,bought_together,subtitle,author
0,All Electronics,HAYLOU PurFree Bone Conduction Headphones Open...,4.0,312,[Open-Ear Comfort : Haylou BC01 bone conductio...,[Open-Ear Bluetooth Bone Conduction Sport Head...,NaN,[{'thumb': 'https://m.media-amazon.com/images/...,"[{'title': 'Haylou BC01', 'url': 'https://www....",HAYLOU,"[Electronics, Headphones, Earbuds & Accessorie...",{'Product Dimensions': '1.77 x 4.92 x 0.39 inc...,B0B1M3DGMQ,NaN,NaN,NaN
1,Computers,Xboun Kids Case for iPad Mini 5 4 3 2 (Old Mod...,4.6,103,[🍎【𝘾𝙤𝙢𝙥𝙖𝙩𝙞𝙗𝙞𝙡𝙞𝙩𝙮 】Designed exclusively for App...,[],10.99,[{'thumb': 'https://m.media-amazon.com/images/...,[{'title': 'LEDNICEKER Kids Case for iPad Mini...,Xboun,"[Electronics, Computers & Accessories, Tablet ...","{'Standing screen display size': '7.9 Inches',...",B08FFYP2BF,NaN,NaN,NaN
2,Cell Phones & Accessories,ohbox USB C to Lightning Cable 3FT 2Pack USB-C...,4.5,544,[【iPhone Fast Charging Cord】The USB C to iphon...,[],13.99,[{'thumb': 'https://m.media-amazon.com/images/...,[{'title': 'The cable works well & solves the ...,ohbox,"[Electronics, Computers & Accessories, Compute...","{'Brand': 'Ohbox', 'Connector Type': 'USB Type...",B0BY8SNST4,NaN,NaN,NaN
3,All Electronics,4 Pairs Galaxy Buds Live Ear Tips Ear Hooks Ki...,4.2,125,[【 Compatibility 】 Compatible with Samsung Gal...,[4 Pairs Galaxy Buds Live Ear Tips Ear Hooks K...,6.99,[{'thumb': 'https://m.media-amazon.com/images/...,[{'title': 'Very comfortable and doesn't make ...,IiEXCEL,"[Electronics, Headphones, Earbuds & Accessorie...",{'Product Dimensions': '2.3 x 1.8 x 0.5 inches...,B0BJ1STN9X,NaN,NaN,NaN
4,Cell Phones & Accessories,Troniker Stylish Case for Galaxy Buds 2 Pro (2...,4.5,1093,[【High Quality Material & Perfect Size】Designe...,[],11.99,[{'thumb': 'https://m.media-amazon.com/images/...,"[{'title': 'Convenient for ear buds! ', 'url':...",Troniker,"[Electronics, Headphones, Earbuds & Accessorie...",{'Product Dimensions': '2.7 x 1.8 x 0.9 inches...,B0BRSTFYHT,NaN,NaN,NaN


In [7]:
list(df_items["features"].items())[2]

(2,
 ['【iPhone Fast Charging Cord】The USB C to iphone charger cord supports a rapid charge up to 3A (max). It also support data transfer rates up to 480Mbps. Note: You need a USB-C port wall charger to match it. (including 18W, 20W, 30W, 60W, or 87W USB-C Power Adapter).',
  '【Safe USB-C Lightning Cable】The USB C Lightning cable is built into the smart chip to ensure 100% compatibility with your Apple device.No warning messages pop up and the latest C94 lightning connector is designed for fast charging.',
  '【High Quality and Ultimate Durability】USBC to lightning cable is built-in overvoltage protection and Lasts 10× longer than other cables and proven to withstand over 10,000 bends in strict laboratory tests.. Heat-resistant connector and premium nylon braided ensure complete safety and reliability.',
  '【Applicable Apple Equipment】USB-C to iPhone cable supports PD Fast Charge 3A (max) for iPhone 14 / 14 Pro/ 14 Pro Max/ 13/ 13 Pro/ 13 Pro Max/ 13 mini, iPhone 12/ 12 Mini/ 12 Pro/ 12 

In [8]:
list(df_items["images"].items())[0]

(0,
 [{'thumb': 'https://m.media-amazon.com/images/I/31iUdKtHsnL._AC_US40_.jpg',
   'large': 'https://m.media-amazon.com/images/I/31iUdKtHsnL._AC_.jpg',
   'variant': 'MAIN',
   'hi_res': 'https://m.media-amazon.com/images/I/51IRfOhD8AL._AC_SL1500_.jpg'},
  {'thumb': 'https://m.media-amazon.com/images/I/41IK0Ux0R3L._AC_US40_.jpg',
   'large': 'https://m.media-amazon.com/images/I/41IK0Ux0R3L._AC_.jpg',
   'variant': 'PT02',
   'hi_res': 'https://m.media-amazon.com/images/I/6191xutgqOL._AC_SL1500_.jpg'},
  {'thumb': 'https://m.media-amazon.com/images/I/41DHkaIvLiL._AC_US40_.jpg',
   'large': 'https://m.media-amazon.com/images/I/41DHkaIvLiL._AC_.jpg',
   'variant': 'PT03',
   'hi_res': 'https://m.media-amazon.com/images/I/71YLn8+0IkL._AC_SL1500_.jpg'},
  {'thumb': 'https://m.media-amazon.com/images/I/41UgtydTITL._AC_US40_.jpg',
   'large': 'https://m.media-amazon.com/images/I/41UgtydTITL._AC_.jpg',
   'variant': 'PT04',
   'hi_res': 'https://m.media-amazon.com/images/I/71h7Bg74irL._AC_SL1

### preprocess title and feature

In [9]:
def preprocess_description(row):
    return f"{row['title']} {' '.join(row['features'])}"

In [10]:
def extract_first_large_image(row):
    return row['images'][0].get("large","")

In [11]:
df_items["description"] = df_items.apply(preprocess_description, axis=1)

df_items["image"] = df_items.apply(extract_first_large_image, axis=1)

In [12]:
df_items.head()

,main_category,title,average_rating,rating_number,features,description,price,images,videos,store,categories,details,parent_asin,bought_together,subtitle,author,image
0,All Electronics,HAYLOU PurFree Bone Conduction Headphones Open...,4.0,312,[Open-Ear Comfort : Haylou BC01 bone conductio...,HAYLOU PurFree Bone Conduction Headphones Open...,NaN,[{'thumb': 'https://m.media-amazon.com/images/...,"[{'title': 'Haylou BC01', 'url': 'https://www....",HAYLOU,"[Electronics, Headphones, Earbuds & Accessorie...",{'Product Dimensions': '1.77 x 4.92 x 0.39 inc...,B0B1M3DGMQ,NaN,NaN,NaN,https://m.media-amazon.com/images/I/31iUdKtHsn...
1,Computers,Xboun Kids Case for iPad Mini 5 4 3 2 (Old Mod...,4.6,103,[🍎【𝘾𝙤𝙢𝙥𝙖𝙩𝙞𝙗𝙞𝙡𝙞𝙩𝙮 】Designed exclusively for App...,Xboun Kids Case for iPad Mini 5 4 3 2 (Old Mod...,10.99,[{'thumb': 'https://m.media-amazon.com/images/...,[{'title': 'LEDNICEKER Kids Case for iPad Mini...,Xboun,"[Electronics, Computers & Accessories, Tablet ...","{'Standing screen display size': '7.9 Inches',...",B08FFYP2BF,NaN,NaN,NaN,https://m.media-amazon.com/images/I/41yiihYpMg...
2,Cell Phones & Accessories,ohbox USB C to Lightning Cable 3FT 2Pack USB-C...,4.5,544,[【iPhone Fast Charging Cord】The USB C to iphon...,ohbox USB C to Lightning Cable 3FT 2Pack USB-C...,13.99,[{'thumb': 'https://m.media-amazon.com/images/...,[{'title': 'The cable works well & solves the ...,ohbox,"[Electronics, Computers & Accessories, Compute...","{'Brand': 'Ohbox', 'Connector Type': 'USB Type...",B0BY8SNST4,NaN,NaN,NaN,https://m.media-amazon.com/images/I/5140l8GD20...
3,All Electronics,4 Pairs Galaxy Buds Live Ear Tips Ear Hooks Ki...,4.2,125,[【 Compatibility 】 Compatible with Samsung Gal...,4 Pairs Galaxy Buds Live Ear Tips Ear Hooks Ki...,6.99,[{'thumb': 'https://m.media-amazon.com/images/...,[{'title': 'Very comfortable and doesn't make ...,IiEXCEL,"[Electronics, Headphones, Earbuds & Accessorie...",{'Product Dimensions': '2.3 x 1.8 x 0.5 inches...,B0BJ1STN9X,NaN,NaN,NaN,https://m.media-amazon.com/images/I/41+Jk-9gMa...
4,Cell Phones & Accessories,Troniker Stylish Case for Galaxy Buds 2 Pro (2...,4.5,1093,[【High Quality Material & Perfect Size】Designe...,Troniker Stylish Case for Galaxy Buds 2 Pro (2...,11.99,[{'thumb': 'https://m.media-amazon.com/images/...,"[{'title': 'Convenient for ear buds! ', 'url':...",Troniker,"[Electronics, Headphones, Earbuds & Accessorie...",{'Product Dimensions': '2.7 x 1.8 x 0.9 inches...,B0BRSTFYHT,NaN,NaN,NaN,https://m.media-amazon.com/images/I/517vqFEpCx...


In [13]:
list(df_items["description"].items())[0]

(0,
 'HAYLOU PurFree Bone Conduction Headphones Open-Ear Bluetooth 5.2 Sport Headphones -IP67 Waterproof Wireless Earphones for Cycling and Running - CVC Dual Microphone Noise Reduction Call Open-Ear Comfort : Haylou BC01 bone conduction sound, comfortable sports wearing experience, healthier ear canal（Use bone conduction technology to receive speech, close to the bone, and the sound wave directly transmits to the auditory nerve through the bone） Safety & Connection : Stay aware and motivated through any workout with our advanced bone conduction technology. Haylou delivers quality audio while leaving your eardrums open to surroundings for ultimate safety. IP67 Waterproof Rated : Completely sweat and waterproof for workouts, fitness and running,cycling.(Not suitable for swimming.) 8 Hours of Music & Clear Calls : Enjoy eight continuous hours of music, calls and podcasts with our Bluetooth headphones. Haylou BC01 also features a 10-minute quick charge for up to 2 hours of battery life. (

#### Sample 50 items from the dataset

In [14]:
df_sample = df_items.sample(n=50, random_state=42)

In [15]:
len(df_sample)

50

In [16]:
data_to_embed = df_sample[[ "description", "image", "rating_number", "price", "average_rating", "parent_asin"]].to_dict(orient="records")

In [17]:
data_to_embed

[{'description': 'PNY 256GB PRO Elite Class 10 U3 V30 microSDXC Flash Memory Card - 100MB/s, Class 10, U3, V30, A2, 4K UHD, Full HD, UHS-I, Micro SD & SanDisk MobileMate USB 3.0 microSD Card Reader- SDDR-B531-GN6NN Product 1: Class 10, U3, V30 speed class performance with read speeds up to 100MB/s, and write speeds up to 90MB/s for fast and smooth burst mode HD Photography and 4K Ultra HD Videography Product 1: A2 App Performance enables apps to run directly from the microSD card, delivering faster app launch and performance. A2 provides minimally 4000 IOPS (Read) and 2000 IOPS (Write) Product 1: Record and transfer videos, photos, music, files and more from microSD enabled host devices such as Android smartphones and tablets, action and surveillance cameras, drones, computers and more Product 1: Included SD adapter for compatibility with SD enabled host devices including DSLR cameras, video cameras, desktops, and laptops Product 2: Compact and durable microSD card reader Product 2: Fa

### Define embedding function

In [ ]:
from openai import OpenAI

client = OpenAI(
    base_url="http://localhost:11434/v1",
    api_key="ollama",  
)


In [35]:
def get_embedding(text, model="nomic-embed-text"):
    safe_text = text[:10000]
    response = client.embeddings.create(
        input=safe_text,
        model=model
    )
    return response.data[0].embedding

In [22]:
get_embedding("This is a sample description for embedding.")

[-0.00948325078934431,
 0.012058141641318798,
 0.027258556336164474,
 -0.05333848297595978,
 -0.00197103270329535,
 -0.01828557625412941,
 -0.026689395308494568,
 -0.020734688267111778,
 0.027556661516427994,
 0.01653590053319931,
 0.03787371143698692,
 -0.002093489747494459,
 0.03336760401725769,
 0.008175315335392952,
 -0.015481525100767612,
 0.003441209439188242,
 -0.019416728988289833,
 -0.05459068715572357,
 -0.03769243136048317,
 -0.04583422467112541,
 -0.04796198010444641,
 0.020276710391044617,
 -0.0745227262377739,
 -0.017317868769168854,
 -0.015847085043787956,
 0.03982369229197502,
 0.039216477423906326,
 0.03783240541815758,
 0.06001000106334686,
 -0.009964903816580772,
 0.002223898656666279,
 0.03785277530550957,
 0.0012547960504889488,
 -0.05744506046175957,
 -0.006924338173121214,
 -0.006462203338742256,
 0.04315252602100372,
 -0.048255693167448044,
 -0.017380185425281525,
 -0.04144918546080589,
 0.011150099337100983,
 0.002850027522072196,
 0.0482356883585453,
 -0.04788

#### Create Qdrant collection

In [23]:
qdrant_client = QdrantClient(url="http://localhost:6333")

/home/nurda/ai-engineering/.venv/lib/python3.12/site-packages/qdrant_client/qdrant_remote.py:288: UserWarning: Failed to obtain server version. Unable to check client-server compatibility. Set check_compatibility=False to skip version check.
  show_warning(


In [41]:
qdrant_client.create_collection(
    collection_name="Amazon_items_collection1",
    vectors_config=VectorParams(
        size=768,
        distance=Distance.COSINE
    )
)   

True

#### Embed data

In [27]:
pointstruct = PointStruct(
    id=0,
    vector=get_embedding("This is a sample description for embedding."),
    payload={
        "text": "This is a sample description for embedding.",
        "model": "text-embedding-3-small",
    }
)

In [28]:
pointstruct

PointStruct(id=0, vector=[-0.00948325078934431, 0.012058141641318798, 0.027258556336164474, -0.05333848297595978, -0.00197103270329535, -0.01828557625412941, -0.026689395308494568, -0.020734688267111778, 0.027556661516427994, 0.01653590053319931, 0.03787371143698692, -0.002093489747494459, 0.03336760401725769, 0.008175315335392952, -0.015481525100767612, 0.003441209439188242, -0.019416728988289833, -0.05459068715572357, -0.03769243136048317, -0.04583422467112541, -0.04796198010444641, 0.020276710391044617, -0.0745227262377739, -0.017317868769168854, -0.015847085043787956, 0.03982369229197502, 0.039216477423906326, 0.03783240541815758, 0.06001000106334686, -0.009964903816580772, 0.002223898656666279, 0.03785277530550957, 0.0012547960504889488, -0.05744506046175957, -0.006924338173121214, -0.006462203338742256, 0.04315252602100372, -0.048255693167448044, -0.017380185425281525, -0.04144918546080589, 0.011150099337100983, 0.002850027522072196, 0.0482356883585453, -0.04788336530327797, -0.0

#### Amazon data

In [42]:
pointstruct = []

for i, data in enumerate(data_to_embed):
    embedding = get_embedding(data["description"])
    pointstruct.append(
        PointStruct(
            id=i,
            vector=embedding,
            payload=data
        )
    )

In [37]:
pointstruct

[PointStruct(id=0, vector=[0.013423208147287369, 0.05709562823176384, -0.17772410809993744, 0.015171516686677933, 0.1015402227640152, -0.05331674963235855, 0.03367140144109726, 0.008494429290294647, -0.01655389927327633, -0.027801621705293655, -0.01265675574541092, 0.048786960542201996, 0.05174693092703819, 0.03933154046535492, -0.03218289092183113, -0.006248887162655592, -0.016131896525621414, -0.04704158008098602, 0.016664749011397362, 0.05148911103606224, -0.008599835447967052, -0.03788148611783981, -0.06307704746723175, -0.05267710983753204, 0.034267526119947433, 0.03648051247000694, 0.002132571768015623, -0.04958544299006462, -0.06469915807247162, 0.007702807430177927, 0.07936262339353561, -0.04484635591506958, -0.06843358278274536, -0.018542218953371048, -0.02708554081618786, -0.048753317445516586, -0.018149811774492264, -0.028110746294260025, -0.019198333844542503, 0.046231064945459366, 0.04992655664682388, -0.021877076476812363, 0.03551079332828522, 0.008130664937198162, 0.0871

In [38]:
len(pointstruct)

50

### Write embedded data to QDrant

In [44]:
qdrant_client.upsert(
    collection_name="Amazon_items_collection1",
    wait=True,
    points=pointstruct
)

UpdateResult(operation_id=1, status=<UpdateStatus.COMPLETED: 'completed'>)

#### define function for data retrieval

In [47]:
def retrieve_data(query, top_k=5):
    query_embedding = get_embedding(query)
    search_result = qdrant_client.query_points(
        collection_name="Amazon_items_collection1",
        query=query_embedding,
        limit=top_k
    )
    return search_result

In [48]:
retrieve_data("What kind of charging cords do you offer?", top_k=10).points

[ScoredPoint(id=40, version=1, score=0.7114527, payload={'description': "Light Up iPhone Charger Cable - MFi Certified Apple RGB Light LED Charging Cord.USB A to Lightning Cable for iPhone,iPad,iPod and More (5ft) This is a nice gift:this kind of light up RGB iphone charger cable, children and the elderly will especially like it. It's also suitable to give to friends or to use by yourself, it will be cool! Very Special:Once the charging cable is plugged into the charger, it will alternately emit blue, red,green,purple,yellow,Pink light. It looks glowing and feels superb. Ultra-fast charging:This LED iPhone charger cable supports data transfer speeds of up to 480Mb/s & fast charging speeds of up to 3 amps. Compatible with iPhone 14,13,12, 11, X, 8, 7, 6, 5 series models and iPad, iPod Touch has good compatibility. High-quality materials:The metal shell is used to obtain an excellent grip. The soft and strong PVC wire makes the LED charging cable wear-resistant, and the bending life exce